In [1]:
import os
import json
from tqdm.auto import tqdm
from pathlib import Path

import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# RUN ONCE: change working directory to project root
cwd = Path.cwd()

pwd = cwd.parent

os.chdir(pwd)
print(f"Changed working directory to {pwd}")

if Path.cwd() != pwd:
    raise RuntimeError(f"Failed to change working directory to {pwd}")

Changed working directory to /Users/jdk/projects/recruiting-reader


In [3]:
# create driver
from src.scraper.driver import make_driver
driver = make_driver(headless=True)

In [ ]:
from src.config import PORTAL_2025, PORTAL_2024
from src.scraper.link_extractor import scrape_portal_player_links


# urls_2024 = scrape_portal_player_links(driver, PORTAL_2024)
# print(f"Found {len(urls_2024)} player transfer portal links for 2024 class.")

urls_2025 = scrape_portal_player_links(driver, PORTAL_2025)
print(f"Found {len(urls_2025)} player transfer portal links for 2025 class.")

with open("data/urls_2025_transfers.json", "w") as f:
    json.dump(urls_2025, f)

# all_urls = list(dict.fromkeys(urls_2025 + urls_2024))
# print("total portal urls:", len(all_urls))

Clicking 'Load More':   0%|          | 0/200 [00:00<?, ?click/s]

Found 3010 player transfer portal links for 2025 class.


In [5]:
# urls_2025[2:]
# first two results erroneous cbssports.com links

In [6]:
## tests
# from src.scraper.player_scraper import scrape_player
# p = scrape_player(driver, 'https://247sports.com/player/emmanuel-pregnon-46140927/college-299861/')
# p

# from src.utils.tests import test_timeline
# events = test_timeline(driver, "https://247sports.com/player/howard-sampson-46129672/college-310950/")
# len(events)

# from src.config import DEBUG
# from src.scraper.player_scraper import scrape_player

# rows = []
# for u in tqdm(urls_2025[2:102]):
#     try:
#         d = scrape_player(driver, u)
#         rows.append(d)
#     except Exception as e:
#         print("FAIL", u, e)

# portal_df = pd.DataFrame(rows)
# portal_df.isna().sum()

In [3]:
with open("data/urls_2025_transfers.json", "r") as f:
    urls_2025 = json.load(f)

In [20]:
from src.storage.cache_new import run_scrape
# from src.config import CACHE_PATH

CACHE_PATH = "data/portal_2025_transfers_1218_run5.jsonl"

all_urls = list(dict.fromkeys(urls_2025[2:]))
scraped_count, cache_count = run_scrape(
    all_urls,
    out_path=CACHE_PATH,
    num_workers=8,        # try 4, 6, 8
    recycle_every=100     # try 50 if you see instability
)
print("scraped this run:", scraped_count, "already cached:", cache_count)


Scraping:   0%|          | 0/10 [00:00<?, ?player/s]

Louis Brown IV commits to Baylor Bears
<re.Match object; span=(15, 25), match='commits to'>
Louis Brown IV entered the transfer portal
None
Louis Brown IV commits to Colorado State Rams
<re.Match object; span=(15, 25), match='commits to'>
Easton Messer commits to Florida Atlantic Owls
<re.Match object; span=(14, 24), match='commits to'>
Easton Messer entered the transfer portal
None
Easton Messer enrolls at Western Kentucky...
<re.Match object; span=(14, 24), match='enrolls at'>
Simon Mapa commits to New Mexico Lobos
<re.Match object; span=(11, 21), match='commits to'>
Simon Mapa entered the transfer portal
None
Simon Mapa enrolls at California Golden Bears
<re.Match object; span=(11, 21), match='enrolls at'>
Dekel Crowdus commits to Wisconsin Badgers
<re.Match object; span=(14, 24), match='commits to'>
Dekel Crowdus entered the transfer portal
None
Dekel Crowdus withdraws from transfer portal
None
Dekel Crowdus entered the transfer portal
None
Dekel Crowdus commits to Hawaii Rainbow W

In [ ]:
# spot checks
# CACHE_PATH = "data/portal_2025_transfers_1218_run5.jsonl"
# from src.storage.cache_new import load_cache
# test = load_cache(CACHE_PATH)

In [21]:
from src.storage.cache_new import load_cache

test = load_cache(CACHE_PATH)
test_df = pd.DataFrame(test).T.set_index('id_247')
test_clean = {
    k: v
    for k, v in test.items()
    if (
        isinstance(v, dict)
        and v.get("pos_247") is not None
        # and v.get("hs_city") is not None
        # and v.get("transfer_origin") is not None
        # and v.get("transfer_destination") is not None
    )
}
test_clean_df = pd.DataFrame(test_clean).T.set_index('id_247')
test_clean_df.to_csv(CACHE_PATH[:-5] + 'csv')

with open(CACHE_PATH, "w") as f:
    for v in test_clean.values():
            f.write(json.dumps(v) + "\n")

In [22]:
len(test_clean)
# missing 7 entries at 3001

3003

In [23]:
test_clean_df.isna().sum()

name                       0
pos_247                    0
hs_name                    5
hs_city                    0
hs_state                   0
transfer_rating            0
transfer_year              0
transfer_ovr_rank         88
transfer_pos_rank         35
transfer_stars             0
transfer_origin           45
transfer_destination      18
hs_class                   1
hs_rating_247           1010
hs_pos                     0
composite_rating        1119
composite_natl_rank     1121
composite_pos_rank      1121
source_hs_url              0
hs_stars                1010
source_player_url          0
transfer_status         2862
dtype: int64

In [24]:
test_clean_df[test_clean_df['hs_name'].isna()]
# drop my problem children

,name,pos_247,hs_name,hs_city,hs_state,transfer_rating,transfer_year,transfer_ovr_rank,transfer_pos_rank,transfer_stars,...,hs_class,hs_rating_247,hs_pos,composite_rating,composite_natl_rank,composite_pos_rank,source_hs_url,hs_stars,source_player_url,transfer_status
id_247,,,,,,,,,,,,,,,,,,,,,
46128460,Nuer Gatkuoth,EDGE,None,Edmonton,AB,86,2025,712,69,3,...,2022,None,LB,None,None,None,https://247sports.com/player/nuer-gatkuoth-461...,None,https://247sports.com/player/nuer-gatkuoth-461...,NaN
46137816,Melvin Siani,OT,None,Ontario,CA,86,2025,717,64,3,...,2023,None,OT,None,None,None,https://247sports.com/player/melvin-siani-4613...,None,https://247sports.com/player/melvin-siani-4613...,NaN
46140445,Vili Taufatofua,EDGE,None,New Zealand,NEW,83,2025,2127,197,3,...,2023,None,DL,None,None,None,https://247sports.com/player/vili-taufatofua-4...,None,https://247sports.com/player/vili-taufatofua-4...,NaN
46158069,Kenton Allen,LB,None,Riverside,CA,83,2025,2256,176,3,...,2023,None,LB,None,None,None,https://247sports.com/player/kenton-allen-4615...,None,https://247sports.com/player/kenton-allen-4615...,NaN
46141053,Charlie Leota,DL,None,Auckland,AC,82,2025,2315,246,3,...,2023,None,DL,None,None,None,https://247sports.com/player/charlie-leota-461...,None,https://247sports.com/player/charlie-leota-461...,NaN


In [ ]:
from src.utils.tests import test_timeline
test_timeline(driver, "https://247sports.com/player/howard-sampson-46129672/college-310950/")